# Inter-Annotator Agreement (IAA) & Human Validation of the Classifier Labels

**Purpose.** Before any analysis of the classified reviews, we must establish that the labels are
trustworthy. This notebook answers two distinct questions, and it is important not to conflate them:

1. **Is the construct well-defined?** → *human–human agreement*: if two trained humans cannot agree on
   what counts as e.g. "Coded" racialisation, no model can be validated against the concept.
2. **Does the model capture it?** → *human–model agreement*: given that humans agree, how often does the
   model's label match theirs?

**The annotators.** Two humans annotated identical samples independently:
- **Ethel** — `data_annotation_validation/ethel_validation_sample.xlsx` (label columns `*_human`)
- **Keziah** — `data_annotation_validation/keziah_validation_sample.xlsx` (label column `keziah_annotation`)

**The tasks validated here** (the `themes` dimension is skipped — its human annotation is incomplete):

| task | sheet(s) | n | sample drawn from | seed |
|---|---|---|---|---|
| social (binary) | `social_recall` | 100 | random from full 1.27M corpus | 42 |
| social (binary) | `social_precision` | 50 | random from model social=**yes** | 42 |
| racialised (binary) | `reclassed_racialised_recall` | 100 | random from all 504k racially-classified (model label mostly **no**) | 42 |
| racialised (binary) | `reclassed_racialised_precision` | 50 | random from corrected racialised=**yes** (7,768) | 42 |
| racialised (binary) | `reclassed_racialised_dimension` | 200 | random from corrected racialised=**yes** (7,768) | 43 |
| approach | `approach_*` | 150 | stratified ~50 per label-combo | 1101 |
| context | `context_validation_sample` | 150 | stratified ~50 per label-combo | 1102 |
| connotation | `connotation_validation_sample` | 248 | stratified ~50 per label-combo | 1103 |

(Provenance: `scripts/generate_validation_samples.py` and `..._reclassed.py`.)

**⚠ Sampling caveat that shapes every number below.** The precision/recall sheets are *conditional*
samples (drawn from model-yes or from the whole classified pool), and the dimension sheets are
*stratified by the model's own label*. Agreement statistics computed on such samples describe those
strata — they are **not** population-level agreement estimates. Where we extrapolate (e.g. corpus-level
false-negative counts), we say so explicitly and carry the uncertainty.

## 1. Load the two annotators' workbooks

We load every sheet pair, keep only `review_id`, the machine label shown to the annotator, and the
human label. We verify that both annotators saw **exactly the same review_ids** — this is what makes
human–human agreement computable at all.

In [ ]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd

BASE = Path('/home/ethel/Desktop/swimming_pool_project/spp_15052026')
VAL  = BASE / 'data_annotation_validation'
OUT  = VAL / 'iaa_results'
OUT.mkdir(exist_ok=True)

ethel  = pd.ExcelFile(VAL / 'ethel_validation_sample.xlsx')
keziah = pd.ExcelFile(VAL / 'keziah_validation_sample.xlsx')

# task registry: (ethel sheet, ethel human col, keziah sheet, machine-label col in sheet)
TASKS = {
    'social_recall':        ('social_recall',                 'social_human',      'social_recall',                 'social'),
    'social_precision':     ('social_precision',              'social_human',      'social_precision',              'social'),
    'racialised_recall':    ('reclassed_racialised_recall',   'racialised_human',  'reclassed_racialised_recall',   'racialised'),
    'racialised_precision': ('reclassed_racialised_precision','racialised_human',  'reclassed_racialised_precision','racialised'),
    'racialised_dimension': ('reclassed_racialised_dimension','racialised_human',  'reclassed_racialised_dimension','racialised'),
    'approach':             ('approach_validation_recalss3',  'aproach__human',    'approach_validation_sample',    'approach'),
    'context':              ('context_validation_sample',     'context_human',     'context_validation_sample',    'context'),
    'connotation':          ('connotation_validation_sample', 'connotation_human', 'connotation_validation_sample', 'connotation'),
}

frames = {}
for task, (esh, ecol, ksh, mcol) in TASKS.items():
    de = ethel.parse(esh)
    dk = keziah.parse(ksh)
    de['review_id'] = de['review_id'].astype(str)
    dk['review_id'] = dk['review_id'].astype(str)
    assert set(de.review_id) == set(dk.review_id), f'{task}: annotators saw different reviews!'
    merged = (de[['review_id', 'trans_review', mcol, ecol]]
                .rename(columns={mcol: 'machine_shown', ecol: 'ethel'})
                .merge(dk[['review_id', 'keziah_annotation']].rename(columns={'keziah_annotation': 'keziah'}),
                       on='review_id', validate='one_to_one'))
    frames[task] = merged
    print(f'{task:22s} n={len(merged):4d}  (ids identical across annotators ✓)')

## 2. Normalise the raw annotations

Free-text annotation always produces formatting noise. Every rule below was decided by inspecting the
raw value counts (shown after the code), and each is intentionally conservative — we fix *formatting*,
never *meaning*:

| rule | example | rationale |
|---|---|---|
| lowercase + strip whitespace | `'no '` → `no`, `'Explicit'` → `explicit` | case/spacing carries no meaning |
| fix known typo | `positve` → `positive` | unambiguous slip |
| split multi-labels on `,` `/` | `'Explicit, Implicit'` → `{explicit, implicit}` | annotators listed multiple labels |
| expand `both` per task | approach → `{explicit, implicit}`; context → `{incident, generalised}`; connotation → `{positive, negative}` | see below |
| values outside the task's label set | `'great'` (social) → **invalid, excluded** | cannot be interpreted |

**The `both` decision (affects Keziah only).** Keziah used the shorthand `both` in three tasks. For
*context* it is unambiguous (only two labels exist). For *approach* her own vocabulary was only
{explicit, implicit, both} — she never used *coded* — so `both` can only mean `{explicit, implicit}`.
For *connotation* she used `neutral` as a separate value, so `both` is read as `{positive, negative}`
(mixed sentiment). These interpretations are recorded here and can be revisited; the affected counts
are printed below.

**Machine labels** in the sheets are stored in several formats (`['Explicit' 'Implicit']` numpy repr,
JSON, bare strings) — the same parser handles all of them.

Every task is normalised to a **set of labels** so that single-label and multi-label answers are
handled uniformly.

In [ ]:
VALID = {
    'social':     {'yes', 'no'},
    'racialised': {'yes', 'no'},
    'approach':   {'explicit', 'coded', 'implicit'},
    'context':    {'incident', 'generalised'},
    'connotation':{'positive', 'negative', 'neutral'},
}
BOTH = {
    'approach':    {'explicit', 'implicit'},
    'context':     {'incident', 'generalised'},
    'connotation': {'positive', 'negative'},
}
TYPOS = {'positve': 'positive', 'posiitve': 'positive', 'negativve': 'negative',
         'negatvie': 'negative', 'generalsed': 'generalised'}

def task_family(task):
    if task.startswith('social'):      return 'social'
    if task.startswith('racialised'):  return 'racialised'
    return task

def norm_human(val, family):
    """Free-text human annotation -> frozenset of valid labels, or None if missing/invalid."""
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return None
    s = str(val).strip().lower()
    if not s:
        return None
    if s == 'both' and family in BOTH:
        return frozenset(BOTH[family])
    parts = [p.strip() for p in re.split(r'[,/]', s) if p.strip()]
    parts = [TYPOS.get(p, p) for p in parts]
    labels = {p for p in parts if p in VALID[family]}
    if not labels or labels != set(parts):        # any unrecognised token -> invalid
        return None if not labels else frozenset(labels)
    return frozenset(labels)

def norm_machine(val, family):
    """Machine label cell -> frozenset of labels.

    The sheets store machine labels in three formats depending on which export they
    came from: numpy repr  ['Crowd' 'Etiquette'],  JSON  ["Explicit"],  and plain
    comma-joined text  Positive, Negative.  Quoted tokens are extracted where present;
    otherwise the string is split on commas/slashes like the human answers.
    """
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return None
    s = str(val).strip()
    if not s:
        return None
    tokens = re.findall(r"'([^']+)'|\"([^\"]+)\"", s) if s.startswith('[') else []
    if tokens:
        labels = [a or b for a, b in tokens]
    else:
        labels = [p for p in re.split(r'[,/]', s.strip('[]')) if p.strip()]
    labels = {TYPOS.get(t.strip().lower(), t.strip().lower()) for t in labels if t.strip()}
    labels &= VALID[family]
    return frozenset(labels) if labels else None

for task, df in frames.items():
    fam = task_family(task)
    df['ethel_n']   = df['ethel'].map(lambda v: norm_human(v, fam))
    df['keziah_n']  = df['keziah'].map(lambda v: norm_human(v, fam))
    df['machine_n'] = df['machine_shown'].map(lambda v: norm_machine(v, fam))

print('Rows dropped per task because an annotation was missing or uninterpretable:')
for task, df in frames.items():
    bad_e = df['ethel_n'].isna().sum()
    bad_k = df['keziah_n'].isna().sum()
    raw_e = df.loc[df['ethel_n'].isna(), 'ethel'].dropna().tolist()
    raw_k = df.loc[df['keziah_n'].isna(), 'keziah'].dropna().tolist()
    print(f'  {task:22s} ethel: {bad_e:2d} {raw_e if raw_e else ""}   keziah: {bad_k:2d} {raw_k if raw_k else ""}')

## 3. Agreement metrics — what we compute and why

**Percent agreement** is intuitive but misleading when one label dominates (two annotators who both
answer *no* 95% of the time agree 90% by pure chance). We therefore report:

- **Cohen's κ** for the binary tasks: the standard chance-corrected agreement for two raters,
  $\kappa = (p_o - p_e)/(1 - p_e)$ where $p_o$ is observed and $p_e$ chance agreement from the raters'
  marginal label frequencies. We attach a **bootstrap 95% CI** (resampling items) because our samples
  are small (50–248 items).
  *Skew caveat:* on sheets where one class is ~96% of items (by sampling design!), κ can look poor
  despite near-perfect raw agreement — the "kappa paradox". We therefore always show κ **next to** raw
  agreement and the confusion table, and never interpret κ alone.
- **For the dimension tasks (labels are *sets*)**: plain κ does not apply to set-valued answers. We report:
  - **exact-set agreement** — strictest: the whole set must match;
  - **mean Jaccard overlap** — $|A \cap B| / |A \cup B|$ per item, credits partial overlap;
  - **per-label Cohen's κ** — each label is treated as its own binary present/absent task, which shows
    *which* categories carry the disagreement.

**Interpretation bands** (Landis & Koch 1977; use as convention, not law):
< 0.20 slight · 0.21–0.40 fair · 0.41–0.60 moderate · 0.61–0.80 substantial · > 0.80 almost perfect.

In [ ]:
def kappa(a, b):
    """Cohen's kappa for two equal-length label sequences (hashable labels)."""
    a, b = pd.Series(list(a)), pd.Series(list(b))
    po = (a == b).mean()
    cats = sorted(set(a) | set(b))
    pe = sum((a == c).mean() * (b == c).mean() for c in cats)
    return np.nan if pe == 1 else (po - pe) / (1 - pe)

def kappa_ci(a, b, n_boot=2000, seed=7):
    rng = np.random.default_rng(seed)
    a, b = np.asarray(list(a), dtype=object), np.asarray(list(b), dtype=object)
    stats = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(a), len(a))
        stats.append(kappa(a[idx], b[idx]))
    lo, hi = np.nanpercentile(stats, [2.5, 97.5])
    return lo, hi

def jaccard(s1, s2):
    return len(s1 & s2) / len(s1 | s2) if (s1 | s2) else np.nan

def wilson_ci(k, n, z=1.96):
    """Wilson score interval for a proportion k/n."""
    if n == 0: return (np.nan, np.nan)
    p = k / n
    d = 1 + z**2 / n
    c = (p + z**2 / (2 * n)) / d
    h = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / d
    return (max(0, c - h), min(1, c + h))

def pairwise(df, c1, c2):
    """Rows where both columns have a usable (non-None) value."""
    m = df[c1].notna() & df[c2].notna()
    return df.loc[m, c1], df.loc[m, c2], int(m.sum())

## 4. Binary task 1 — `social`

The `social_recall` sheet is a **random sample of the whole corpus**, so it estimates corpus-level
agreement and lets us cross-tabulate humans against the model on both classes. The `social_precision`
sheet contains only model-*yes* reviews, so the human yes-rate there **is** a precision estimate for the
model's *yes* label.

In [ ]:
def binary_report(task, df):
    fam = task_family(task)
    # sets -> scalars for binary tasks ('yes'/'no'); multi answers impossible here
    for c in ('ethel_n', 'keziah_n', 'machine_n'):
        df[c + 's'] = df[c].map(lambda s: next(iter(s)) if isinstance(s, frozenset) and len(s) == 1 else None)
    rows = []
    for name, (c1, c2) in {'ethel vs keziah': ('ethel_ns', 'keziah_ns'),
                           'ethel vs model':  ('ethel_ns', 'machine_ns'),
                           'keziah vs model': ('keziah_ns', 'machine_ns')}.items():
        a, b, n = pairwise(df, c1, c2)
        k = kappa(a, b); lo, hi = kappa_ci(a, b)
        rows.append({'comparison': name, 'n': n, 'raw_agreement': (a == b).mean(),
                     'kappa': k, 'kappa_ci': f'[{lo:.2f}, {hi:.2f}]'})
    rep = pd.DataFrame(rows)
    print(f'=== {task} ===')
    display(rep.round(3))
    a, b, _ = pairwise(df, 'ethel_ns', 'keziah_ns')
    display(pd.crosstab(a.rename('ethel'), b.rename('keziah')))
    return rep

social_recall_rep    = binary_report('social_recall', frames['social_recall'])
social_precision_rep = binary_report('social_precision', frames['social_precision'])

In [ ]:
# Precision of the model's social='yes' (from the precision sheet, model label is 'yes' throughout)
df = frames['social_precision']
print('Estimated precision of social=yes (human agrees the review is social):')
for who in ('ethel_ns', 'keziah_ns'):
    d = df[df[who].notna()]
    k, n = (d[who] == 'yes').sum(), len(d)
    lo, hi = wilson_ci(k, n)
    print(f'  {who[:-3]:8s}: {k}/{n} = {k/n:.1%}   (95% Wilson CI {lo:.1%}–{hi:.1%})')

# From the random recall sheet: full human-vs-model confusion on both classes
df = frames['social_recall']
for who in ('ethel_ns', 'keziah_ns'):
    a, b, n = pairwise(df, who, 'machine_ns')
    print(f'\n{who[:-3]} vs model (random corpus sample, n={n}):')
    display(pd.crosstab(a.rename('human'), b.rename('model')))

## 5. Binary task 2 — `racialised`  *(the label the whole study rests on)*

Three sheets contribute:
- `racialised_precision` (50) and `racialised_dimension` (200) are both drawn from the **corrected
  racialised = yes** set and test the same thing (does a human agree this review is racialised) --
  pooled into one n=250 sample below, both for the precision estimate and for the ethel-vs-keziah
  IAA/kappa report, rather than reporting the two sheets' kappas separately (their small/skewed
  individual n's gave contradictory-looking numbers, 0.658 vs -0.023).
- `racialised_recall` (100) is a random sample of **all 504k racially-classified** reviews, i.e. almost
  entirely model-*no* → the human yes-rate among model-*no* rows estimates the **false-negative rate**,
  the quantity that determines how much racialisation the pipeline *misses*.

**Extrapolation warning.** The FN rate is estimated from ~100 reviews; a rate of 2–3% may look small
but multiplies into thousands of reviews across a ~500k corpus, and its confidence interval is wide.
We report Wilson CIs and translate them into corpus-level ranges explicitly.

In [ ]:
# precision and dimension sheets are both drawn from the same population (corrected
# racialised=yes, n=7,768) and test the same thing -- does a human agree this review is
# racialised -- so we pool them into a single n=250 sample rather than reporting two
# separate, noisier kappas (the split gave contradictory-looking kappas, 0.658 vs -0.023,
# purely from each half's small/skewed n -- the kappa-paradox caveat above applies harder
# at n=50/200 than at the pooled n=250).
rac_pool_df = pd.concat([frames['racialised_precision'], frames['racialised_dimension']], ignore_index=True)
rac_precdim_rep = binary_report('racialised_precision_dimension (pooled)', rac_pool_df)
rac_rec_rep = binary_report('racialised_recall', frames['racialised_recall'])


In [ ]:
# ---- pooled precision on n=250 from corrected racialised=yes ----
# reuses rac_pool_df from the cell above (already has ethel_ns/keziah_ns/machine_ns
# from binary_report's pooled call) instead of re-concatenating from scratch.
pool = rac_pool_df
print('Precision of final racialised=yes  (pooled precision+dimension sheets, all drawn from corrected yes):')
for who in ('ethel_ns', 'keziah_ns'):
    d = pool[pool[who].notna()]
    k, n = (d[who] == 'yes').sum(), len(d)
    lo, hi = wilson_ci(k, n)
    print(f'  {who[:-3]:8s}: {k}/{n} = {k/n:.1%}   (95% Wilson CI {lo:.1%}–{hi:.1%})')
both = pool[pool['ethel_ns'].notna() & pool['keziah_ns'].notna()]
k = ((both['ethel_ns'] == 'yes') & (both['keziah_ns'] == 'yes')).sum()
lo, hi = wilson_ci(k, len(both))
print(f'  both agree yes: {k}/{len(both)} = {k/len(both):.1%}   (95% CI {lo:.1%}–{hi:.1%})')

In [ ]:
# ---- false-negative rate among model-no (recall sheet) and corpus-level meaning ----
df = frames['racialised_recall']
model_no = df[df['machine_ns'] == 'no']
print(f'model-no rows in recall sample: {len(model_no)} of {len(df)}')
N_MODEL_NO_IN_CORPUS = 496_304 - 3_024   # final 'no' labels in the verified corpus that were model-'no'
for who in ('ethel_ns', 'keziah_ns'):
    d = model_no[model_no[who].notna()]
    k, n = (d[who] == 'yes').sum(), len(d)
    lo, hi = wilson_ci(k, n)
    print(f'\n  {who[:-3]}: flags {k}/{n} model-no reviews as racialised = {k/n:.1%}  (CI {lo:.1%}–{hi:.1%})')
    print(f'     → corpus-level missed racialised reviews ≈ {k/n*N_MODEL_NO_IN_CORPUS:,.0f}'
          f'  (CI {lo*N_MODEL_NO_IN_CORPUS:,.0f}–{hi*N_MODEL_NO_IN_CORPUS:,.0f})')
print('\n⚠ Compare these magnitudes to the 7,768 detected racialised reviews: even a ~1–2% FN rate implies')
print('   the detected set may be a MINORITY of all racialised reviews. If the CI cannot be narrowed')
print('   (larger recall annotation), prevalence claims must be framed as lower bounds, or corrected')
print('   (e.g. Rogan–Gladen) with the uncertainty carried through.')

## 6. Dimension tasks — `approach`, `context`, `connotation`

Human and machine answers are **sets of labels**. For human–model agreement we compare against the
**current canonical labels** from `analysis_10072026/racialised_reviews_analysis.parquet` (not the
possibly-outdated label printed in the annotation sheet — where the two differ we report it).

A per-label κ table shows *which* categories drive disagreement (e.g. is `coded` confusable with
`implicit`?). Disagreement rows are exported to CSV for qualitative inspection.

In [ ]:
canon = pd.read_parquet(BASE / 'analysis_10072026' / 'gmaps_data' / 'racialised_reviews_analysis.parquet',
                        columns=['review_id', 'approach', 'context', 'connotation'])
canon['review_id'] = canon['review_id'].astype(str)
canon = canon.set_index('review_id')

def canon_set(v):
    if v is None or (isinstance(v, float) and pd.isna(v)): return None
    return frozenset(str(x).strip().lower() for x in list(v))

def dimension_report(task):
    df = frames[task].copy()
    dim = task_family(task)
    df['canonical_n'] = df['review_id'].map(canon[dim].map(canon_set))
    drift = (df['canonical_n'] != df['machine_n']).sum()
    print(f'=== {task} ===  (sheet label ≠ current canonical label on {drift}/{len(df)} rows)')

    rows = []
    for name, (c1, c2) in {'ethel vs keziah':     ('ethel_n', 'keziah_n'),
                           'ethel vs model':      ('ethel_n', 'canonical_n'),
                           'keziah vs model':     ('keziah_n', 'canonical_n')}.items():
        a, b, n = pairwise(df, c1, c2)
        exact = (a == b).mean()
        jac = np.mean([jaccard(x, y) for x, y in zip(a, b)])
        rows.append({'comparison': name, 'n': n, 'exact_set_agreement': exact, 'mean_jaccard': jac})
    rep = pd.DataFrame(rows)
    display(rep.round(3))

    # per-label kappa (label present/absent), human vs human and (pooled) human vs model
    lab_rows = []
    for lab in sorted(VALID[dim]):
        a, b, n = pairwise(df, 'ethel_n', 'keziah_n')
        k_hh = kappa([lab in s for s in a], [lab in s for s in b])
        a2, b2, _ = pairwise(df, 'ethel_n', 'canonical_n')
        k_em = kappa([lab in s for s in a2], [lab in s for s in b2])
        a3, b3, _ = pairwise(df, 'keziah_n', 'canonical_n')
        k_km = kappa([lab in s for s in a3], [lab in s for s in b3])
        lab_rows.append({'label': lab, 'κ ethel-keziah': k_hh, 'κ ethel-model': k_em, 'κ keziah-model': k_km})
    display(pd.DataFrame(lab_rows).round(3))

    # export disagreements for qualitative review
    dis = df[(df['ethel_n'].notna()) & (df['keziah_n'].notna()) & (df['ethel_n'] != df['keziah_n'])]
    out = OUT / f'disagreements_{dim}.csv'
    dis_out = dis[['review_id', 'trans_review', 'ethel', 'keziah', 'machine_shown']].copy()
    dis_out['canonical'] = dis['canonical_n'].map(lambda s: sorted(s) if isinstance(s, frozenset) else None)
    dis_out.to_csv(out, index=False)
    print(f'{len(dis)} human–human disagreements exported → {out.name}\n')
    return rep

approach_rep    = dimension_report('approach')
context_rep     = dimension_report('context')
connotation_rep = dimension_report('connotation')

### 6b. `approach` — focus on Ethel vs. model only

Keziah's `approach` sheet turned out to predate the current 3-category scheme: it has no `coded` label at all (only `explicit`/`implicit`/`both`), and even restricting to the subset where both annotators used only `explicit`/`implicit` (no `coded`, no multi-label — 98 rows), agreement was still weak (45.9%, κ=0.07) — a genuine definitional gap in how `explicit`/`implicit` are being applied, not just the missing `coded` category. Keziah is redoing this dimension against the current scheme, so **`ethel vs keziah` and `keziah vs model` for `approach` should be treated as stale until that redo lands** — everything below focuses on `ethel vs model`, which used a consistent scheme throughout.

In [ ]:
df = frames['approach'].copy()
df['canonical_n'] = df['review_id'].map(canon['approach'].map(canon_set))

# per-label precision/recall (treating Ethel's label as ground truth, model as prediction)
rows = []
for lab in sorted(VALID['approach']):
    e = df['ethel_n'].map(lambda s: lab in s if isinstance(s, frozenset) else np.nan)
    m = df['canonical_n'].map(lambda s: lab in s if isinstance(s, frozenset) else np.nan)
    both_present = e.notna() & m.notna()
    e, m = e[both_present], m[both_present]
    tp = ((e) & (m)).sum()
    fp = ((~e) & (m)).sum()
    fn = ((e) & (~m)).sum()
    tn = ((~e) & (~m)).sum()
    precision = tp / (tp + fp) if (tp + fp) else np.nan
    recall = tp / (tp + fn) if (tp + fn) else np.nan
    rows.append({'label': lab, 'n_ethel_used': e.sum(), 'n_model_used': m.sum(),
                 'precision': precision, 'recall': recall, 'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn})
print("Ethel (ground truth) vs. model, per-label precision/recall — approach:")
display(pd.DataFrame(rows).round(3))

mismatch = df[df['ethel_n'] != df['canonical_n']].copy()
mismatch['ethel_label'] = mismatch['ethel_n'].map(lambda s: sorted(s) if isinstance(s, frozenset) else None)
mismatch['model_label'] = mismatch['canonical_n'].map(lambda s: sorted(s) if isinstance(s, frozenset) else None)
out_cols = ['review_id', 'trans_review', 'ethel_label', 'model_label']
mismatch[out_cols].to_csv(OUT / 'disagreements_approach_ethel_vs_model.csv', index=False)
print(f"\n{len(mismatch)} ethel-vs-model disagreements exported -> disagreements_approach_ethel_vs_model.csv")
display(mismatch[out_cols].head(10))


**Takeaway.** Ethel-vs-model agreement on `approach` is good but not uniform across labels. `explicit` has excellent precision (0.99) but middling recall (0.88) — when the model says explicit it's almost always right, but it misses 15 cases Ethel called explicit. `coded` is moderate both ways (precision 0.62, recall 0.81). `implicit` has poor precision (0.39, 16 false positives against only 10 true positives) despite decent recall (0.77). The 28 disagreement rows make the pattern concrete: most of them are cases where **Ethel said `explicit` and the model said `implicit`** — i.e. the model isn't randomly wrong, it has a specific, consistent tendency to call boundary cases `implicit` when a human would call them `explicit`. That's a more actionable finding for refining the classifier or the label guidelines than an aggregate agreement number would suggest, and it's independent of the Keziah scheme-mismatch issue above.

## 7. Summary and reading guide

The table below gathers the headline numbers. How to read it:

- **Human–human is the ceiling.** The model cannot be meaningfully "better" than the human agreement
  on the same items; if human–model ≈ human–human, the model is as consistent as a trained person.
- **Binary κ values on the skewed sheets** (racialised precision/recall) are expected to be low even at
  97% raw agreement — the class balance is degenerate *by sampling design*. Rely on the
  precision/FN-rate estimates with their CIs instead.
- **Dimensions**: exact-set agreement is strict; Jaccard shows how partial the disagreements are. The
  per-label κ tables above identify the categories that need better definitions (or merging) before the
  substantive analysis.

Everything is saved to `data_annotation_validation/iaa_results/` for citation in the paper.

In [ ]:
summary = pd.concat(
    [df.assign(task=t) for t, df in {
        'social_recall': social_recall_rep, 'social_precision': social_precision_rep,
        'racialised_precision_dimension': rac_precdim_rep,
        'racialised_recall': rac_rec_rep,
        'approach': approach_rep, 'context': context_rep, 'connotation': connotation_rep,
    }.items()],
    ignore_index=True,
)
cols = ['task', 'comparison', 'n', 'raw_agreement', 'kappa', 'kappa_ci', 'exact_set_agreement', 'mean_jaccard']
summary = summary[[c for c in cols if c in summary.columns]]
summary.to_csv(OUT / 'iaa_summary.csv', index=False)
display(summary.round(3))
print(f'Saved → {OUT / "iaa_summary.csv"}')